# N007 · 字典、集合、元组与哈希键

**目标：** 选择集合做去重、字典做映射并理解可哈希性。

**先修：** N005。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** dict/set/tuple；键唯一性；计数；成员查询；平均复杂度的条件。

**配套讲解来源：** [同名逐章意见](../../comment/007_dict_set_tuple.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两个高频到"几乎是刷题alphabet"的问题：**查重**和**计数**，并借此学会三个最重要的内置容器——字典（dict）、集合（set）、元组（tuple）。

**问题 1（判断重复）**：给你一个整数数组 nums，判断是否存在某个值至少出现两次。具体例子：输入 `[1,2,1]`，输出 `True`——因为 1 出现了两次（下标 0 和下标 2）；输入 `[1,2,3]`，输出 `False`——三个数各不相同。注意题目只要"有没有"，不问哪个数、在哪里重复，所以只需要"出现过"这一种信息。

**问题 2（频次统计）**：给你一个数组，返回一个"值 → 出现次数"的字典。具体例子：输入 `[2,1,2,3]`，输出 `{2:2, 1:1, 3:1}`——2 出现两次，1 和 3 各一次。这道题要的不只是"有没有"，而是"每个值各有多少个"，所以集合不够用，得上字典。

这两道题合起来教一个选型直觉：**只关心"是否出现过"用集合；要保留"出现过多少次"用字典**。另外本章还会讲清"什么样的东西能当键"（可哈希性），这是新手从字典走向哈希表类题目的必过关卡。

## 2. 基础知识：先读懂这些词

- **字典（dict）**：形如 `{键: 值}` 的容器，靠键直接查值，就像用学号查学生。`counts[2]` 意思是"键 2 对应的值"。同一个键只能存一份，后写的值会覆盖先写的（键唯一性）。
- **集合（set）**：只有键没有值的容器，元素不重复。`seen.add(x)` 把 x 放进去；如果 x 已经在，集合保持不变。给集合添加重复元素不会报错、也不会变长。
- **元组（tuple）**：圆括号包住的不可变序列，如 `(2,3)`。它最常见的用途是当字典键或集合元素（比如把坐标 (行,列) 当键），这是列表做不到的。
- **可哈希（hashable）**：能当键/集合元素的前提。一个对象可哈希，意思是它能被算出一个"身份号码"（哈希值），并且生命周期里这个号码不变。整数、字符串、内容全是可哈希元素的元组都行；**列表是可变的，改一下内容身份就变了，所以不能直接当键**——你拿一个以后可能变化的东西当索引，字典内部就乱了。
- **成员查询（`in`）**：`x in seen` 问"x 在不在这个集合里"。对集合和字典的 `in` 平均只要常数时间；对列表的 `in` 要从头扫到尾，是线性时间。这个差距是本章算法快慢的分水岭。
- **`counts.get(x, 0)`**：从字典取键 x 的值；如果 x 还不存在，不报错而是返回默认值 0。它是计数代码"一行顶四行"的关键。
- **平均复杂度**：哈希查找"通常"是常数时间，这个结论是在数据不刻意制造冲突的前提下谈的平均情况；如果有人恶意构造大量碰撞键，最坏会退化。算法题一般按平均 O(1) 使用。
- **前缀（prefix）**：数组从开头到某个位置为止的那一段，比如 `[2,1,2,3]` 的前 3 个元素组成前缀 `[2,1,2]`。本章的正确性论证就是"处理完前 i 个元素后，容器里恰好是这 i 个元素的信息"。

## 3. 思路推导：从小例子开始

**思路 A：判断重复（contains_duplicate）。**

- **Step 1：选容器。** 只需要"这个值出现过没有"，用集合 seen。每读一个数，先查在不在，再把它记进去。
- **Step 2：一边扫一边查。** 对每个 x：如果 `x in seen`，说明以前见过它，立刻返回 True（早退，后面不用看了）；否则 `seen.add(x)` 继续扫。
- **Step 3：注意"先查后加"的顺序。** 必须先查询再把自己加进去。如果反过来先加再查，那么查的时候 x 一定在集合里（自己刚放进去），会把"第一次出现"误判成"重复"。
- **Step 4：扫完没发现重复返回 False。** 空数组自然落在这一步，返回 False。

**手算演示一轮（nums=[1,2,1]）**：初始 seen 为空。第 1 轮 x=1：`1 in seen`？空集合，不在，把 1 加入，seen={1}。第 2 轮 x=2：不在 seen 里，加入，seen={1,2}。第 3 轮 x=1：`1 in seen`？在！（第 1 轮加过）立刻返回 True。全程 3 次循环，答案正确。

**思路 B：频次统计（count_values）。**

- **Step 1：选容器。** 需要记录"每个值各多少次"，键是值、值是次数，用字典 counts。
- **Step 2：逐个累加。** 对每个 x 执行 `counts[x] = counts.get(x,0) + 1`：x 以前没见过就取默认 0 再加 1（第一次记 1）；见过就取旧次数加 1。
- **Step 3：扫完返回整个字典。**

**手算演示一轮（nums=[2,1,2,3]，与 “运行示例”部分 表格一致）**：初始 counts={}。x=2：get(2,0)=0，存 2:1，得 {2:1}。x=1：存 1:1，得 {2:1,1:1}。x=2：get(2)=1，加 1 存 2:2，得 {2:2,1:1}。x=3：存 3:1，最终 {2:2,1:1,3:1}。“运行示例”部分 的表格用的输入是 `[2,1,2,3,2]`，它逐行列出了每个前缀的频次字典：空前缀 `{}` → `[2]` 得 `{2:1}` → `[2,1]` 得 `{2:1,1:1}` → `[2,1,2]` 得 `{2:2,1:1}` → `[2,1,2,3]` 得 `{2:2,1:1,3:1}` → 全部 `[2,1,2,3,2]` 得 `{2:3,1:1,3:1}`。你可以对照着看 counts 是怎么一格一格长出来的。

## 4. 核心代码：contains_duplicate

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def contains_duplicate(nums: list[int]) -> bool:
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False
```

### 逐段读懂代码

**核心实现（“分段实现”部分）对应实现：**

（对应程序见下方分段实现与完整代码。）

**函数一：`contains_duplicate`**

- `seen=set()` —— 建一个空集合，当"已见过的值"的账本。set 里每个值只存一份，天然去重。
- `for x in nums:` —— 从头到尾逐个看数组元素。
- `if x in seen:` / `return True` —— 先查询（在加入自己**之前**查）。x 已经在账本里，说明它这次是第二次或之后的出现，数组里有重复，立刻返回 True。提前返回让函数在发现答案的那一轮就结束，后面的元素不再处理。
- `seen.add(x)` —— 查过没查到才走到这行：把 x 记进账本，供后面的元素比对。顺序（先查后加）正是为了避免"x 和自己匹配"的假阳性。
- `return False` —— 循环走完都没触发过 return True，说明每个值第一次出现时账本里都没有它，即所有值互不相同，返回 False。nums 为空列表时循环体一次不执行，直接走到这行。

**函数二：`count_values`**

- `def count_values(nums: list[int]) -> dict[int,int]:` —— 类型注解 `dict[int,int]` 读作"键是整数、值也是整数的字典"，正对应"值 → 次数"。
- `counts={}` —— 从空字典起步，账上还没有任何键。
- `for x in nums:` —— 逐个元素处理。
- `counts[x]=counts.get(x,0)+1` —— 本章最浓缩的一行，拆开看：`counts.get(x,0)` 取 x 的旧次数，x 不存在时安全地取 0；`+1` 在旧次数上加一；`counts[x]=...` 把新次数写回键 x。它一行同时处理了"x 是新键"和"x 是旧键"两种情况，不需要 if/else 分叉。
- `return counts` —— 循环结束，字典里每个键的值恰好是该键在 nums 里的出现总次数。

**小实例表格的生成代码（“运行示例”部分）：** “运行示例”部分 开头是与其他章相同的 `show_table` 展示辅助函数（escape 转安全文本、拼 HTML 表格、display 渲染，不含算法），实验部分就两行：

```python
a = [2, 1, 2, 3, 2]

show_table(['已读前缀', '频次'], [(a[:i], count_values(a[:i])) for i in range(len(a) + 1)])
```

- `a=[2,1,2,3,2]` —— 演示数组特意让 2 出现三次、1 和 3 各一次，这样频次表里能同时看到"新键从 0 到 1"和"旧键往上累加"两种变化。
- `range(len(a)+1)` —— i 取 0 到 5 共 6 个值。前缀数比元素数多一个，是因为要包含"空前缀"这一行：它对应"一个元素都还没读"的初始状态，频次自然是空字典 `{}`。
- `a[:i]` —— 切片，每轮复制出一个新列表当"已读前缀"，各行的前缀互不干扰。
- `count_values(a[:i])` —— 每一行都**真的调用**函数、对前缀从头算一遍频次。注意这不是增量更新：表格展示的是"前缀长什么样，频次表就该长什么样"的对应关系，正好把第三节那个前缀不变量逐行可视化出来。五行非空前缀各算一遍，总代价约 1+2+3+4+5=15 次计数操作，演示规模下毫无压力。

第三节已经把六行的频次值逐行列过，这里再补一个观察角度：盯住键 2 的值那一列，它走的是 1→1→2→2→3——2 每次再出现就加一，1 和 3 的键纹丝不动。这一列就是 `counts[x]=counts.get(x,0)+1` 在时间轴上的样子。

## 5. 分段实现：按顺序运行

**本章接口：** `contains_duplicate(nums)`、`count_values(nums)`

### contains_duplicate

In [1]:
def contains_duplicate(nums: list[int]) -> bool:
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False

### count_values

In [2]:
def count_values(nums: list[int]) -> dict[int, int]:
    counts = {}
    for x in nums:
        counts[x] = counts.get(x, 0) + 1
    return counts

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[2,1,2,3,2]
show_table(['已读前缀','频次'], [(a[:i],count_values(a[:i])) for i in range(len(a)+1)])

已读前缀,频次
[],{}
[2],{2: 1}
"[2, 1]","{2: 1, 1: 1}"
"[2, 1, 2]","{2: 2, 1: 1}"
"[2, 1, 2, 3]","{2: 2, 1: 1, 3: 1}"
"[2, 1, 2, 3, 2]","{2: 3, 1: 1, 3: 1}"


## 7. 正确性、适用条件与复杂度

处理新元素前，seen 恰好是已读元素的集合；counts[x] 恰好是该前缀中 x 的出现次数。查询再更新避免把当前元素与自己重复匹配。

**代价：** 平均 O(n) 时间，O(u) 空间，u 为不同值数量。

### 展开理解

**contains_duplicate 为什么对：** 我们盯住一个从头到尾成立的性质——**在处理下一个新元素之前，seen 恰好是"已经读过的那一段元素"构成的集合**。一开始 seen 是空集，对应"还没读过任何元素"，成立。每一轮：我们先查询再添加。如果查到了，说明当前这个 x 在更早的位置出现过（因为 seen 里只有更早的元素），重复确实存在，返回 True 名正言顺；如果没查到，我们把 x 加进去，seen 又恰好变成了"读到这里为止"的集合，性质保持。如果整个循环都平安走完，说明每个元素第一次到来时账本里都没有它，也就是没有值出现两次以上，返回 False 正确。"先查后加"这个顺序还顺便保证了当前元素不会和自己匹配——查的时候自己还没进账本。

**count_values 为什么对：** 同样盯前缀性质——**处理完前 i 个元素后，counts[x] 恰好等于 x 在这前 i 个元素里的出现次数**。每一轮把 x 的计数从"前 i-1 个里的次数"加 1 变成"前 i 个里的次数"，其他键不动。归纳着往前滚，处理完全部 n 个元素后，counts 就是整个数组的准确频次表。两个函数都遍历一遍数组、每轮做固定的常数次哈希操作，没有遗漏也没有重复计数。

**复杂度是多少：** 两个函数都是平均 O(n) 时间：n 个元素各处理一次，每次的集合/字典查询与写入平均是常数开销。空间是 O(u)，u 是不同值的个数（集合或字典最多存 u 个键）。拿具体数字感受：n=10⁵ 的数组大约十万次哈希操作，毫秒级跑完；对比暴力做法（两层循环逐对比较，约 n²/2 = 50 亿次），快了五个数量级。要注意 O(1) 查询是**平均情况**的说法，前提是键的分布没有人为制造大量哈希碰撞；日常刷题按平均情况理解即可。

**空间账本用具体数字看：** 空间是 O(u) 而不是 O(n)，u 是不同值的个数。n=10⁵ 的数组若值全互不相同，seen 要装 10⁵ 个键（这个量级的 Python 集合大约占几 MB 内存）；若 10⁵ 个元素全是 7，seen 从头到尾只有 `{7}` 一个键，O(1) 空间。换句话说："数组多长"决定时间，"值多杂"决定空间，这两个维度要分开估。

**顺带解释一个你可能注意到的现象：** 频次字典打印出来键的顺序是 2、1、3，这是 Python 3.7 起字典按**插入顺序**保留键的结果——2 在数组里最先出现，所以排最前。“自动测试”部分 那条守恒断言用 `sum(...values())` 做检查而不逐键比对字典，恰好绕开了对键序的依赖，测试因此更稳。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 逐条看：

```python
assert contains_duplicate([1, 2, 1]) is True

assert contains_duplicate([1, 2, 3]) is False

assert contains_duplicate([]) is False

assert sum(count_values([2, 1, 2, 3]).values()) == 4

assert count_values([]) == {}

print('N007: 所有本章断言通过')
```

- `assert contains_duplicate([1,2,1]) is True`：正常值 + 正例。1 在首尾各出现一次，必须返回 True。用 `is True` 而非 `== True` 是严格身份检查，确认函数返回的是货真价实的布尔值 True 而不是碰巧为真的其他对象。
- `assert contains_duplicate([1,2,3]) is False`：正常值 + 反例。三个值互不相同，必须走完整个循环落到最后的 `return False`，验证"无重复"路径。
- `assert contains_duplicate([]) is False`：空数组边界。循环体一次不执行，直接返回 False——"空数组里没有重复"这个退化情形不需要特殊分支，天然被最后的 return 覆盖。
- `assert sum(count_values([2,1,2,3]).values()) == 4`：**守恒检查**。频次字典是 `{2:2, 1:1, 3:1}`，`.values()` 取出所有次数 `[2,1,1]`，求和等于 4——恰好是原数组长度。这条聪明在它不逐键比对（那会把字典的键序也测进去），只验证"总次数守恒"：如果计数逻辑漏数或多数任何一个元素，总和不等于 n=4，立刻失败。总和对、且每个键计数非负，基本锁定了计数正确性。
- `assert count_values([]) == {}`：空输入边界。空数组应该安静返回空字典，不报错。
- 最后的 print：前面全部通过才会打印，代表整组测试通过。

**如果实现写错了，哪条断言会先响？** 做个故障推演，帮你建立"断言 = 报警器"的直觉：

- 把"先查后加"倒过来（先 `seen.add(x)` 再查 `x in seen`）：每个元素一进循环就把自己加进账本，查询必然命中自己，`contains_duplicate([1,2,3])` 会错误地返回 True，第二条断言立刻挂——第三节强调的顺序问题，在这里能看到它是真的会被测试抓住。
- 把 `counts.get(x,0)` 忘写默认值 0（写成 `counts.get(x)+1`）：第一次遇到新键，get 返回 None，`None+1` 直接抛 TypeError，第四条断言以异常方式失败。
- 把累加写成覆盖（`counts[x]=1`）：每个键的值都是 1，`sum(count_values([2,1,2,3]).values())` 算出来是 3（不同值个数）而不是 4，守恒检查立刻把它揪出来——这条断言恰好是为"忘记累加"这类 bug 准备的。
- 第一条断言用 `is True` 而不是 `== True` 抓的是另一类问题：如果实现写成 `return 1`（真值但不是布尔），`1 == True` 在 Python 里成立、会蒙混过关，`1 is True` 不成立、当场拦下。`is` 守的是"返回货真价实的布尔值"这个接口契约。

**把五条断言分个层：**

| 断言 | 层次 | 它守护的契约 |
| --- | --- | --- |
| `contains_duplicate([1,2,1]) is True` | 正常值正例 | 有重复 → 返回真布尔 True |
| `contains_duplicate([1,2,3]) is False` | 正常值反例 | 无重复时走完全程返回 False |
| `contains_duplicate([]) is False` | 边界 | 空数组天然无重复 |
| `sum(count_values([2,1,2,3]).values()) == 4` | 守恒 | 总次数等于数组长度 |
| `count_values([]) == {}` | 边界 | 空输入安静返回空字典 |

In [4]:
assert contains_duplicate([1, 2, 1]) is True

assert contains_duplicate([1, 2, 3]) is False

assert contains_duplicate([]) is False

assert sum(count_values([2, 1, 2, 3]).values()) == 4

assert count_values([]) == {}

print('N007: 所有本章断言通过')

N007: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 不用set完成去重。

**练习 2：** 解释列表不能直接作为字典键。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（不用 set 完成去重/查重）**：提示——两条路可走。路一（暴力）：双层循环，对每个 i 检查后面的 j 是否 `nums[i] == nums[j]`，正确但 O(n²)，写完可以拿它当"对拍器"和集合版互相验证。路二（有序化）：`nums.sort()` 或 `sorted(nums)` 之后，重复元素必然相邻，只需一次线性扫描比较相邻两个元素。手算示例用 `[1,2,1]` 期望 True；边界用 `[]`（False）和单元素 `[7]`（False）。做完对比三种方法的时间量级，体会集合换来的是什么。
- **练习 2（解释列表不能直接作为字典键）**：提示——从"可哈希 = 身份号终身不变"讲起：字典靠哈希值决定键存在哪个桶里；列表内容可变，今天 `[1,2]` 放进桶 A，明天改成 `[1,3]`，按新哈希去找就再也找不回旧条目了。动手验证：`d = {[1,2]: 'x'}` 会直接抛 `TypeError: unhashable type: 'list'`。修复办法是把列表转成元组 `d[tuple([1,2])] = 'x'`（元组不可变、可哈希）；要注意元组作键要求其元素本身也可哈希——`(1,[2,3])` 同样不行。手算示例可以用坐标计数：把 `[(0,0),(1,0),(0,0)]` 三个"点"转成元组当键，得到 `{(0,0):2,(1,0):1}`。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def contains_duplicate(nums: list[int]) -> bool:
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False

def count_values(nums: list[int]) -> dict[int, int]:
    counts = {}
    for x in nums:
        counts[x] = counts.get(x, 0) + 1
    return counts

# 示例与回归测试
assert contains_duplicate([1, 2, 1]) is True

assert contains_duplicate([1, 2, 3]) is False

assert contains_duplicate([]) is False

assert sum(count_values([2, 1, 2, 3]).values()) == 4

assert count_values([]) == {}

print('N007: 所有本章断言通过')

N007: 所有本章断言通过


## 11. 代表题与迁移

- **217. Contains Duplicate**：判断数组是否有重复值，就是本章 `contains_duplicate` 的原题。它练的是"用集合做已见账本 + 先查后加的扫描模式 + 平均 O(n) 换掉暴力 O(n²)"这一整套组合拳，是后续所有哈希表题（两数之和、分组、最长连续序列等）的地基。

（补充说明：本章 “题目映射”部分 只列了 217 一道代表题；字典计数模式 `counts[x]=counts.get(x,0)+1` 在后续章节（如词频、异位词分组类题目）会反复出现，值得现在就练到肌肉记忆。）

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。